In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


In [1]:
from pyspark.sql import functions as F

player_metrics = spark.table("lh_gold_game.player_metrics")
sessions = spark.table("lh_silver_game.sessions_clean")

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 3, Finished, Available, Finished, False)

In [2]:
d7_flags = (
    sessions
    .groupBy("player_id")
    .agg(
        F.max(
            F.when(
                F.col("days_since_install") == 7,
                1
            ).otherwise(0)
        ).alias("d7_retained")
    )
)

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 4, Finished, Available, Finished, False)

In [3]:
experiment_base = (
    player_metrics
    .join(
        d7_flags,
        on="player_id",
        how="left"
    )
    .fillna({
        "d7_retained": 0
    })
)

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 5, Finished, Available, Finished, False)

In [4]:
product_experiment_metrics = (
    experiment_base
    .groupBy("experiment_group")
    .agg(
        F.count("*").alias("players"),
        F.avg("d7_retained").alias("d7_retention"),
        F.avg("total_sessions").alias("avg_sessions"),
        F.avg("max_level_reached").alias("avg_max_level"),
        F.avg("is_payer").alias("payer_conversion"),
        F.avg("total_revenue").alias("revenue_per_player")
    )
)

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 6, Finished, Available, Finished, False)

In [5]:
display(product_experiment_metrics)

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f69cafe6-a53b-4c75-b2a6-6efcd1f79995)

In [6]:
d7_test = (
    experiment_base
    .groupBy("experiment_group")
    .agg(
        F.count("*").alias("n"),
        F.sum("d7_retained").alias("retained")
    )
)

display(d7_test)

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 078a7e86-15e2-4e51-8a0d-0f470c13cf43)

In [7]:
from statsmodels.stats.proportion import proportions_ztest
import numpy as np

control_retained = 5490
control_n = 25082

treatment_retained = 5651
treatment_n = 24918

counts = np.array([
    treatment_retained,
    control_retained
])

nobs = np.array([
    treatment_n,
    control_n
])

z_stat, p_value = proportions_ztest(
    count=counts,
    nobs=nobs
)

print("Z-statistic:", z_stat)
print("P-value:", p_value)

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 9, Finished, Available, Finished, False)

Z-statistic: 2.1229508656541562
P-value: 0.033757969392275204


In [8]:
control_rate = control_retained / control_n
treatment_rate = treatment_retained / treatment_n

absolute_difference = treatment_rate - control_rate
relative_lift = absolute_difference / control_rate

print("Control D7:", control_rate)
print("Treatment D7:", treatment_rate)
print("Absolute difference:", absolute_difference)
print("Relative lift:", relative_lift)

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 10, Finished, Available, Finished, False)

Control D7: 0.2188820668208277
Treatment D7: 0.22678385103138293
Absolute difference: 0.007901784210555235
Relative lift: 0.03610064691605581


In [9]:
import math

se = math.sqrt(
    (treatment_rate * (1 - treatment_rate) / treatment_n) +
    (control_rate * (1 - control_rate) / control_n)
)

ci_low = absolute_difference - 1.96 * se
ci_high = absolute_difference + 1.96 * se

print("95% CI lower:", ci_low)
print("95% CI upper:", ci_high)

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 11, Finished, Available, Finished, False)

95% CI lower: 0.0006065416526993715
95% CI upper: 0.015197026768411098


In [10]:
payer_test = (
    experiment_base
    .groupBy("experiment_group")
    .agg(
        F.count("*").alias("n"),
        F.sum("is_payer").alias("payers")
    )
)

display(payer_test)

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bb26b4ce-f892-464c-8c22-cd7b992062af)

In [11]:
control_payers = 971
control_n = 25082

treatment_payers = 892
treatment_n = 24918

counts = np.array([
    treatment_payers,
    control_payers
])

nobs = np.array([
    treatment_n,
    control_n
])

z_stat_payer, p_value_payer = proportions_ztest(
    count=counts,
    nobs=nobs
)

print("Z-statistic:", z_stat_payer)
print("P-value:", p_value_payer)

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 13, Finished, Available, Finished, False)

Z-statistic: -1.7210977526970648
P-value: 0.08523308703337017


In [12]:
control_payer_rate = control_payers / control_n
treatment_payer_rate = treatment_payers / treatment_n

payer_absolute_difference = (
    treatment_payer_rate - control_payer_rate
)

payer_relative_change = (
    payer_absolute_difference / control_payer_rate
)

print("Control payer conversion:", control_payer_rate)
print("Treatment payer conversion:", treatment_payer_rate)
print("Absolute difference:", payer_absolute_difference)
print("Relative change:", payer_relative_change)

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 14, Finished, Available, Finished, False)

Control payer conversion: 0.03871302129016825
Treatment payer conversion: 0.03579741552291516
Absolute difference: -0.002915605767253089
Relative change: -0.07531330983959009


In [13]:
payer_se = math.sqrt(
    (treatment_payer_rate * (1 - treatment_payer_rate) / treatment_n) +
    (control_payer_rate * (1 - control_payer_rate) / control_n)
)

payer_ci_low = payer_absolute_difference - 1.96 * payer_se
payer_ci_high = payer_absolute_difference + 1.96 * payer_se

print("95% CI lower:", payer_ci_low)
print("95% CI upper:", payer_ci_high)

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 15, Finished, Available, Finished, False)

95% CI lower: -0.006235413123423132
95% CI upper: 0.00040420158891695413


In [14]:
players = spark.table("lh_silver_game.players_clean")
sessions = spark.table("lh_silver_game.sessions_clean")
player_metrics = spark.table("lh_gold_game.player_metrics")
marketing = spark.table("lh_silver_game.marketing_spend_clean")

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 16, Finished, Available, Finished, False)

In [15]:
creative_retention_flags = (
    sessions
    .groupBy("player_id")
    .agg(
        F.max(
            F.when(F.col("days_since_install") == 7, 1).otherwise(0)
        ).alias("d7_retained"),

        F.max(
            F.when(F.col("days_since_install") == 30, 1).otherwise(0)
        ).alias("d30_retained")
    )
)

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 17, Finished, Available, Finished, False)

In [16]:
creative_player_base = (
    players
    .filter(
        F.col("creative_experiment_group").isin(
            "Creative A",
            "Creative B"
        )
    )
    .select(
        "player_id",
        "creative_experiment_group"
    )
    .join(
        creative_retention_flags,
        on="player_id",
        how="left"
    )
    .join(
        player_metrics.select(
            "player_id",
            "is_payer",
            "total_revenue"
        ),
        on="player_id",
        how="left"
    )
    .fillna({
        "d7_retained": 0,
        "d30_retained": 0,
        "is_payer": 0,
        "total_revenue": 0.0
    })
)

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 18, Finished, Available, Finished, False)

In [17]:
creative_quality_metrics = (
    creative_player_base
    .groupBy("creative_experiment_group")
    .agg(
        F.count("*").alias("players"),
        F.avg("d7_retained").alias("d7_retention"),
        F.avg("d30_retained").alias("d30_retention"),
        F.avg("is_payer").alias("payer_conversion"),
        F.avg("total_revenue").alias("revenue_per_player")
    )
)

display(creative_quality_metrics)

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 46a982a7-d865-4a4e-9225-c32334ec29c2)

In [18]:
creative_group_map = (
    players
    .filter(
        F.col("creative_experiment_group").isin(
            "Creative A",
            "Creative B"
        )
    )
    .select(
        "creative_id",
        "creative_experiment_group"
    )
    .distinct()
)

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 20, Finished, Available, Finished, False)

In [19]:
creative_marketing_ab = (
    marketing
    .join(
        creative_group_map,
        on="creative_id",
        how="inner"
    )
)

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 21, Finished, Available, Finished, False)

In [20]:
creative_funnel_metrics = (
    creative_marketing_ab
    .groupBy("creative_experiment_group")
    .agg(
        F.sum("impressions").alias("impressions"),
        F.sum("clicks").alias("clicks"),
        F.sum("installs").alias("installs"),
        F.sum("spend_usd").alias("spend_usd")
    )
    .withColumn(
        "ctr",
        F.col("clicks") / F.col("impressions")
    )
    .withColumn(
        "cvr",
        F.col("installs") / F.col("clicks")
    )
    .withColumn(
        "cpi",
        F.col("spend_usd") / F.col("installs")
    )
)

display(creative_funnel_metrics)

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 94ea0ff8-087c-404e-82e9-9389b83a214d)

In [21]:
creative_experiment_metrics = (
    creative_funnel_metrics
    .join(
        creative_quality_metrics,
        on="creative_experiment_group",
        how="inner"
    )
)

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 23, Finished, Available, Finished, False)

In [22]:
creative_experiment_metrics = (
    creative_experiment_metrics
    .withColumn(
        "total_revenue",
        F.col("revenue_per_player") * F.col("players")
    )
    .withColumn(
        "roas",
        F.col("total_revenue") / F.col("spend_usd")
    )
)

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 24, Finished, Available, Finished, False)

In [23]:
display(creative_experiment_metrics)

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, b412748d-ff14-4f7d-8e2a-f8f6782764cf)

In [24]:
creative_d7_test = (
    creative_player_base
    .groupBy("creative_experiment_group")
    .agg(
        F.count("*").alias("n"),
        F.sum("d7_retained").alias("retained")
    )
)

display(creative_d7_test)

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 26, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5c2a6a9a-b321-4982-a562-1ffae1f74011)

In [25]:
creative_a_retained = 872
creative_a_n = 4830

creative_b_retained = 764
creative_b_n = 4830

counts = np.array([
    creative_a_retained,
    creative_b_retained
])

nobs = np.array([
    creative_a_n,
    creative_b_n
])

z_stat_creative_d7, p_value_creative_d7 = proportions_ztest(
    count=counts,
    nobs=nobs
)

print("Z-statistic:", z_stat_creative_d7)
print("P-value:", p_value_creative_d7)

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 27, Finished, Available, Finished, False)

Z-statistic: 2.929713882758858
P-value: 0.003392742368453522


In [26]:
creative_a_rate = creative_a_retained / creative_a_n
creative_b_rate = creative_b_retained / creative_b_n

creative_d7_difference = creative_a_rate - creative_b_rate
creative_d7_relative_lift = creative_d7_difference / creative_b_rate

print("Creative A D7:", creative_a_rate)
print("Creative B D7:", creative_b_rate)
print("Absolute difference:", creative_d7_difference)
print("Relative lift:", creative_d7_relative_lift)

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 28, Finished, Available, Finished, False)

Creative A D7: 0.1805383022774327
Creative B D7: 0.15817805383022773
Absolute difference: 0.022360248447204967
Relative lift: 0.1413612565445026


In [27]:
creative_d7_se = math.sqrt(
    (creative_a_rate * (1 - creative_a_rate) / creative_a_n) +
    (creative_b_rate * (1 - creative_b_rate) / creative_b_n)
)

creative_d7_ci_low = creative_d7_difference - 1.96 * creative_d7_se
creative_d7_ci_high = creative_d7_difference + 1.96 * creative_d7_se

print("95% CI lower:", creative_d7_ci_low)
print("95% CI upper:", creative_d7_ci_high)

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 29, Finished, Available, Finished, False)

95% CI lower: 0.007407726142132663
95% CI upper: 0.03731277075227727


In [28]:
creative_payer_test = (
    creative_player_base
    .groupBy("creative_experiment_group")
    .agg(
        F.count("*").alias("n"),
        F.sum("is_payer").alias("payers")
    )
)

display(creative_payer_test)

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 30, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0d627c90-68e7-4027-8bab-7eaf0f96fb04)

In [29]:
creative_a_payers = 135
creative_a_n = 4830

creative_b_payers = 96
creative_b_n = 4830

counts = np.array([
    creative_a_payers,
    creative_b_payers
])

nobs = np.array([
    creative_a_n,
    creative_b_n
])

z_stat_creative_payer, p_value_creative_payer = proportions_ztest(
    count=counts,
    nobs=nobs
)

print("Z-statistic:", z_stat_creative_payer)
print("P-value:", p_value_creative_payer)

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 31, Finished, Available, Finished, False)

Z-statistic: 2.5972536329597125
P-value: 0.009397250596259281


In [30]:
creative_a_payer_rate = creative_a_payers / creative_a_n
creative_b_payer_rate = creative_b_payers / creative_b_n

creative_payer_difference = (
    creative_a_payer_rate - creative_b_payer_rate
)

creative_payer_relative_lift = (
    creative_payer_difference / creative_b_payer_rate
)

print("Creative A payer conversion:", creative_a_payer_rate)
print("Creative B payer conversion:", creative_b_payer_rate)
print("Absolute difference:", creative_payer_difference)
print("Relative lift:", creative_payer_relative_lift)

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 32, Finished, Available, Finished, False)

Creative A payer conversion: 0.027950310559006212
Creative B payer conversion: 0.01987577639751553
Absolute difference: 0.008074534161490683
Relative lift: 0.40625


In [31]:
creative_payer_se = math.sqrt(
    (creative_a_payer_rate * (1 - creative_a_payer_rate) / creative_a_n) +
    (creative_b_payer_rate * (1 - creative_b_payer_rate) / creative_b_n)
)

creative_payer_ci_low = (
    creative_payer_difference - 1.96 * creative_payer_se
)

creative_payer_ci_high = (
    creative_payer_difference + 1.96 * creative_payer_se
)

print("95% CI lower:", creative_payer_ci_low)
print("95% CI upper:", creative_payer_ci_high)

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 33, Finished, Available, Finished, False)

95% CI lower: 0.0019832691456757737
95% CI upper: 0.014165799177305594


In [33]:
from pyspark.sql import Row

product_rows = [
    Row(
        experiment_name="Daily Reward Redesign",
        experiment_group="Control",
        primary_metric="d7_retention",
        primary_metric_value=float(control_rate),
        payer_conversion=float(control_payer_rate),
        p_value_primary=float(p_value),
        ci_low_primary=float(ci_low),
        ci_high_primary=float(ci_high)
    ),
    Row(
        experiment_name="Daily Reward Redesign",
        experiment_group="Treatment",
        primary_metric="d7_retention",
        primary_metric_value=float(treatment_rate),
        payer_conversion=float(treatment_payer_rate),
        p_value_primary=float(p_value),
        ci_low_primary=float(ci_low),
        ci_high_primary=float(ci_high)
    )
]

product_gold = spark.createDataFrame(product_rows)

display(product_gold)

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 35, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, eb7f0586-35ba-4dd7-abe3-065568d1116c)

In [34]:
creative_rows = [
    Row(
        experiment_name="Creative Hook Experiment",
        experiment_group="Creative A",
        primary_metric="d7_retention",
        primary_metric_value=float(creative_a_rate),
        payer_conversion=float(creative_a_payer_rate),
        p_value_primary=float(p_value_creative_d7),
        ci_low_primary=float(creative_d7_ci_low),
        ci_high_primary=float(creative_d7_ci_high)
    ),
    Row(
        experiment_name="Creative Hook Experiment",
        experiment_group="Creative B",
        primary_metric="d7_retention",
        primary_metric_value=float(creative_b_rate),
        payer_conversion=float(creative_b_payer_rate),
        p_value_primary=float(p_value_creative_d7),
        ci_low_primary=float(creative_d7_ci_low),
        ci_high_primary=float(creative_d7_ci_high)
    )
]

creative_gold = spark.createDataFrame(creative_rows)

display(creative_gold)

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 36, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5eae5a90-b1e9-4dea-bc45-2e6395d9789a)

In [35]:
experiment_metrics = (
    product_gold
    .unionByName(creative_gold)
)

display(experiment_metrics)

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 37, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9aea05d5-5b24-4812-abf1-15a5edd34403)

In [36]:
experiment_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("lh_gold_game.experiment_metrics")

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 38, Finished, Available, Finished, False)

In [37]:
df_check = spark.table("lh_gold_game.experiment_metrics")

print("Saved row count:", df_check.count())
display(df_check)

StatementMeta(, f55fc3c3-b046-4ac4-a9b0-6f2493543bc9, 39, Finished, Available, Finished, False)

Saved row count: 4


SynapseWidget(Synapse.DataFrame, 58e7c422-9f2d-4ec3-b84c-e66a7ab059b5)